# 00 — Baseline univariado: pH da estação EF01, regime anual (treino 2024)
Baselines clássicos no mesmo desenho travado (`L=8640 → H=288`), agora em 1 ano de treino. Sem holdout interno: val = 4 fatias de 10 dias (uma por estação); 2025 intocado (benchmark no 08). O sazonal lag-365 é incalculável sem 2023 — estreia no benchmark.

In [1]:
import pickle
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados" / "treino").exists())
CSV = ROOT / "dados/treino/ef01-mogi-das-cruzes_ph_2024.csv"
OUT = ROOT / "univariavel" / "resultados" / "00-baseline-ph"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

L, H = 8640, 288
SEASON = 288
INTERP_LIMIT = 24
# val: 4 fatias sazonais (S1 parcial no OD — ver cobertura impressa)
VAL_SLICES = [("2024-04-19", "2024-04-28"), ("2024-07-20", "2024-07-29"),
              ("2024-09-15", "2024-09-24"), ("2024-11-20", "2024-11-24")]
ARIMA_ORDER = (2, 1, 2)
ARIMA_STRIDE = 48
print("ROOT:", ROOT, "| CSV existe:", CSV.exists())

ROOT: /home/marcos/temporal-model | CSV existe: True


## 1. Carga

In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
df = df.rename(columns={"Data hora": "ds", "pH": "y"}).sort_values("ds").reset_index(drop=True)
print(df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()

(105121, 2) 2024-01-01 00:00:00 → 2024-12-31 00:00:00
faltantes: 11665 (11.1%)


,ds,y
count,105121,93456.00000
mean,2024-07-01 12:00:00,5.90729
min,2024-01-01 00:00:00,5.21000
25%,2024-04-01 06:00:00,5.71000
50%,2024-07-01 12:00:00,5.92000
75%,2024-09-30 18:00:00,6.09000
max,2024-12-31 00:00:00,6.68000
std,NaN,0.30356


## 2. EDA — perfil, faltantes e ciclo diário

In [3]:
isna = df["y"].isna().to_numpy()
gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
print(f"maior gap: {gaps.max()} passos = {gaps.max()*5/60:.1f} h | gaps > 24 passos: {(gaps > 24).sum()}")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.3)
ax[0].set_title("ph EF01 2024 — série completa (treino)")
ax[0].set_ylabel("ph")
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("Ciclo diário")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva")

maior gap: 4971 passos = 414.2 h | gaps > 24 passos: 9


fig salva


## 3. Limpeza — grade completa + interpolação limitada

In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_raw = df.set_index("ds")["y"].reindex(idx)
print(f"slots na grade: {len(s_raw)} | linhas no CSV: {len(df)}")
s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")
gi = np.where(s.isna().to_numpy())[0]
blocos = np.split(gi, np.where(np.diff(gi) > 1)[0] + 1) if len(gi) else []
print(f"blocos NaN pós-interp: {len(blocos)}")
for g in blocos:
    print(f"  outage {s.index[g[0]]} → {s.index[g[-1]]} ({len(g)} slots = {len(g)*5/60:.1f} h)")

amostra = slice("2024-09-09", "2024-09-16")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_raw[amostra].index, s_raw[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 09–16/09")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")

slots na grade: 105121 | linhas no CSV: 105121
NaN após interpolação (limite 24): 6408
blocos NaN pós-interp: 9
  outage 2024-01-16 11:35:00 → 2024-01-18 17:50:00 (652 slots = 54.3 h)
  outage 2024-02-13 17:00:00 → 2024-02-13 17:05:00 (2 slots = 0.2 h)
  outage 2024-03-11 10:55:00 → 2024-03-11 11:55:00 (13 slots = 1.1 h)
  outage 2024-04-29 11:30:00 → 2024-05-02 03:10:00 (765 slots = 63.8 h)
  outage 2024-05-27 12:00:00 → 2024-06-13 16:10:00 (4947 slots = 412.2 h)
  outage 2024-10-19 06:05:00 → 2024-10-19 06:30:00 (6 slots = 0.5 h)
  outage 2024-10-19 13:35:00 → 2024-10-19 13:50:00 (4 slots = 0.3 h)
  outage 2024-11-25 15:15:00 → 2024-11-25 15:20:00 (2 slots = 0.2 h)
  outage 2024-12-03 13:30:00 → 2024-12-03 14:50:00 (17 slots = 1.4 h)
fig salva


## 4. Estacionariedade (ADF) e decomposição STL (trecho limpo jul–ago)

In [5]:
trecho = s.loc["2024-07-15":"2024-08-31"].dropna()
stat, pval, *_ = adfuller(trecho.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(trecho.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")

ADF stat=-1.79 p-valor=0.384 → NÃO estacionária


fig salva


## 5. Janelamento + val em 4 fatias sazonais (2025 intocado)

In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
ed = ends.date
is_val = np.zeros(len(ends), dtype=bool)
for a, b in VAL_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    m = (ed >= d0) & (ed <= d1)
    is_val |= m
    print(f"fatia {a} → {b}: {int(m.sum())} janelas válidas")
va = np.where(is_val)[0]
tr = np.where(~is_val)[0]
print(f"treino: {len(tr)} janelas | val: {len(va)} janelas | descartadas (NaN): {len(s)-L-H+1-len(X)}")
assert len(va) > 1000, "val pequena demais — reposicionar fatias!"
daily_mask = (ends.time == pd.Timestamp("23:55").time()) & is_val
daily_idx = np.where(daily_mask)[0]
print("dias-âncora na val:", len(daily_idx),
      [str(ends[i].date()) for i in daily_idx[:5]], "...", [str(ends[i].date()) for i in daily_idx[-5:]])
s_train = s.copy()
for a, b in VAL_SLICES:
    s_train.loc[a:b] = np.nan  # série de ajuste: fatias de val removidas (Prophet/ARIMA)
print("slots de treino p/ ajuste:", int(s_train.notna().sum()))

fatia 2024-04-19 → 2024-04-28: 2880 janelas válidas
fatia 2024-07-20 → 2024-07-29: 2880 janelas válidas
fatia 2024-09-15 → 2024-09-24: 2880 janelas válidas
fatia 2024-11-20 → 2024-11-24: 1440 janelas válidas
treino: 24659 janelas | val: 10080 janelas | descartadas (NaN): 61455
dias-âncora na val: 35 ['2024-04-19', '2024-04-20', '2024-04-21', '2024-04-22', '2024-04-23'] ... ['2024-11-20', '2024-11-21', '2024-11-22', '2024-11-23', '2024-11-24']
slots de treino p/ ajuste: 88633


## 6. Baselines baratos (treino rolante + val)

In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

Xtr, Ytr, Xva, Yva = X[tr], Y[tr], X[va], Y[va]
pred_tr = cheap_preds(Xtr)
pred_va = cheap_preds(Xva)
print("treino rolante:")
print(pd.DataFrame({m: metricas(Ytr, p) for m, p in pred_tr.items()}).T.round(4).to_string())
print("val rolante:")
print(pd.DataFrame({m: metricas(Yva, p) for m, p in pred_va.items()}).T.round(4).to_string())

treino rolante:


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0740  0.1040  1.2896  1.2913
sazonal_naive_288  0.0627  0.0937  1.0927  1.0960
media_movel_288    0.0668  0.0965  1.1616  1.1651
val rolante:


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0564  0.0809  0.9779  0.9774
sazonal_naive_288  0.0421  0.0614  0.7325  0.7320
media_movel_288    0.0468  0.0636  0.8100  0.8103


## 7. ARIMA em grade horária (val com stride — custo)

In [8]:
hs = s.resample("1h").mean()

def arima_hora(e):
    he = e.floor("h")
    ctx = hs.loc[he - pd.Timedelta(hours=719):he].values
    fc = ARIMA(ctx, order=ARIMA_ORDER).fit().get_forecast(24).predicted_mean.values
    return np.repeat(fc, 12)[:H]

def roda_arima(idxs, nome):
    P = np.empty((len(idxs), H))
    t0 = time.time()
    for j, i in enumerate(idxs):
        try:
            P[j] = arima_hora(ends[i])
        except Exception:
            P[j] = np.repeat(X[i, -1], H)
        if (j + 1) % 10 == 0:
            print(f"  {nome}: {j+1}/{len(idxs)} origens...", flush=True)
    print(f"ARIMA {nome}: {len(idxs)} origens em {time.time()-t0:.0f}s")
    return P

idx_a = va[::ARIMA_STRIDE]
Pa = roda_arima(idx_a, "val")
print("ARIMA val:", metricas(Y[idx_a], Pa))
Pd = roda_arima(daily_idx, "val-diaria")
print("ARIMA dias-âncora:", metricas(Y[daily_idx], Pd))

h_tr = s_train.resample("1h").mean().dropna().iloc[-720:].values
with open(OUT / "modelos" / "arima212_cauda_treino.pkl", "wb") as f:
    pickle.dump(ARIMA(h_tr, order=ARIMA_ORDER).fit(), f)
print("modelo salvo")

  val: 10/210 origens...


  val: 20/210 origens...


  val: 30/210 origens...


  val: 40/210 origens...


  val: 50/210 origens...


...

  val-diaria: 10/35 origens...


  val-diaria: 20/35 origens...


  val-diaria: 30/35 origens...


ARIMA val-diaria: 35 origens em 27s
ARIMA dias-âncora: {'MAE': 0.06376712018140586, 'RMSE': 0.08924158441103901, 'MAPE': 1.1154093592743695, 'sMAPE': 1.1056912096677685}


modelo salvo


## 8. Prophet (opcional — pula se `prophet`/CmdStan indisponível; ajuste só no treino)

In [9]:
PROPHET_OK = False
try:
    from prophet import Prophet
    import cmdstanpy
    assert cmdstanpy.cmdstan_path() is not None
    df_train = pd.DataFrame({"ds": s_train.index, "y": s_train.values}).dropna()
    m = Prophet(daily_seasonality=True, weekly_seasonality=True)
    m.fit(df_train)
    fmap = m.predict(pd.DataFrame({"ds": s.index})).set_index("ds")["yhat"]

    def fatia(idxs):
        E = ends[idxs]
        return np.stack([[fmap.loc[d - pd.Timedelta(minutes=5*(H-1-h))] for h in range(H)] for d in E])

    Pp_va, Pp_d = fatia(va), fatia(daily_idx)
    PROPHET_OK = True
    print("Prophet val:", metricas(Yva, Pp_va))
    from prophet.serialize import model_to_json
    (OUT / "modelos" / "prophet_ph.json").write_text(model_to_json(m))
    print("modelo salvo")
except Exception as e:
    print(f"Prophet pulado ({type(e).__name__}: {str(e)[:150]}).")

Importing plotly failed. Interactive plots will not work.


15:22:54 - cmdstanpy - INFO - Chain [1] start processing


15:24:30 - cmdstanpy - INFO - Chain [1] done processing


Prophet val: {'MAE': 0.17031676856849184, 'RMSE': 0.20010077161241327, 'MAPE': 2.9670368600302814, 'sMAPE': 2.9418308517615523}


modelo salvo


## 9. Comparação final + val dia a dia

In [10]:
linhas = {m: metricas(Yva, p) for m, p in pred_va.items()}
linhas["arima_212_h"] = metricas(Y[idx_a], Pa)
if PROPHET_OK:
    linhas["prophet"] = metricas(Yva, Pp_va)
tab_va = pd.DataFrame(linhas).T.round(4)
tab_va.to_csv(OUT / "metricas_val.csv")
tab_tr = pd.DataFrame({m: metricas(Ytr, p) for m, p in pred_tr.items()}).T.round(4)
tab_tr.to_csv(OUT / "metricas_treino.csv")
print("=== treino rolante ===")
print(tab_tr.to_string())
print("=== val rolante ===")
print(tab_va.to_string())

Yd = Y[daily_idx]
diario = {m: metricas(Yd, cheap_preds(X[daily_idx])[m]) for m in pred_tr}
diario["arima_212_h"] = metricas(Yd, Pd)
if PROPHET_OK:
    diario["prophet"] = metricas(Yd, Pp_d)
tab_d = pd.DataFrame(diario).T.round(4)
tab_d.to_csv(OUT / "metricas_val_diaria.csv")
print("=== val dias-âncora ===")
print(tab_d.to_string())

por_dia = pd.DataFrame(
    {m: [mae(Yd[k:k+1], cheap_preds(X[daily_idx])[m][k:k+1]) for k in range(len(Yd))]
     for m in pred_tr},
    index=[str(ends[i].date()) for i in daily_idx])
por_dia["arima_212_h"] = [mae(Yd[k:k+1], Pd[k:k+1]) for k in range(len(Yd))]
if PROPHET_OK:
    por_dia["prophet"] = [mae(Yd[k:k+1], Pp_d[k:k+1]) for k in range(len(Yd))]
por_dia.to_csv(OUT / "metricas_por_dia.csv")
print(por_dia.round(4).to_string())
print(f"\nRégua treino: {tab_tr['MAE'].idxmin()} = {tab_tr['MAE'].min():.4f}")
print(f"Régua val: {tab_va['MAE'].idxmin()} = {tab_va['MAE'].min():.4f}")

=== treino rolante ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0740  0.1040  1.2896  1.2913
sazonal_naive_288  0.0627  0.0937  1.0927  1.0960
media_movel_288    0.0668  0.0965  1.1616  1.1651
=== val rolante ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0564  0.0809  0.9779  0.9774
sazonal_naive_288  0.0421  0.0614  0.7325  0.7320
media_movel_288    0.0468  0.0636  0.8100  0.8103
arima_212_h        0.0565  0.0822  0.9818  0.9811
prophet            0.1703  0.2001  2.9670  2.9418
=== val dias-âncora ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0638  0.0892  1.1154  1.1057
sazonal_naive_288  0.0421  0.0614  0.7322  0.7317
media_movel_288    0.0458  0.0619  0.7921  0.7925
arima_212_h        0.0638  0.0892  1.1154  1.1057
prophet            0.1800  0.2094  3.1365  3.1085
            persistencia  sazonal_naive_288  media_movel_288  arima_212_h  prophet
2024-04-19        0.0305             0.0262       

## 10. Figuras

In [11]:
E = ends[tr]
ks = [0, len(Xtr) // 2, -1]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
for ax, k in zip(axes, ks):
    tc = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H+2015)), E[k] - pd.Timedelta(minutes=5*H), freq="5min")
    ax.plot(tc, Xtr[k][-2016:], lw=0.8, label="contexto (cauda 7d)")
    tf = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H-1)), E[k], freq="5min")
    ax.plot(tf, Ytr[k], "k-", lw=1.5, label="real")
    ax.plot(tf, pred_tr["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, pred_tr["persistencia"][k], ":", lw=1, label="persistência")
    ax.set_title(f"origem {E[k]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

fig, ax = plt.subplots(figsize=(8, 4))
tab_va["MAE"].sort_values().plot.barh(ax=ax)
ax.set_title("MAE na val — baselines (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

fig, ax = plt.subplots(figsize=(12, 3.5))
pdf = por_dia
for col, ls in [("sazonal_naive_288", "--"), ("persistencia", ":"), ("media_movel_288", "-."), ("arima_212_h", "-")]:
    if col in pdf.columns:
        ax.plot(pd.to_datetime(pdf.index), pdf[col], ls, lw=1.1, label=col)
ax.set_title("ph — MAE por dia-âncora na val (4 fatias sazonais)")
ax.legend(fontsize=8); fig.autofmt_xdate()
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-val-dias.png")
print("figs salvas")

figs salvas


## 11. Conclusões
Réguas do regime anual (treino 2024) acima. O sazonal lag-365 estreia no benchmark 2025 (08). Próximo: LSTNet nativo (02).